# Demo 07. Gaussian Quadrature from Orthogonal Polynomials

**Module 4** (quadrature), section 4.3. Uses **SymPy** for the polynomial construction and **mpmath** for high-precision nodes and weights.

Newton-Cotes rules fix the nodes on an equispaced grid and solve for weights. Gaussian quadrature instead chooses both nodes and weights to maximize the degree of exactness. The optimal nodes turn out to be the roots of the degree-$n$ orthogonal polynomial for the interval, and with them an $n$-point rule integrates every polynomial of degree up to $2n-1$ exactly. This demo builds the Legendre nodes and weights from the orthogonal polynomials, verifies the $2n-1$ exactness, and compares the accuracy against composite Simpson at equal cost.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp
import sympy as sp
from ipywidgets import interact, Dropdown, IntSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)
mp.mp.dps = 40   # high-precision nodes and weights

## Why the orthogonal-polynomial roots

On $[-1,1]$ the Legendre polynomials $P_n$ satisfy $\int_{-1}^{1} P_m P_n\,dx = 0$ for $m\neq n$. Take the $n$ roots $x_1,\dots,x_n$ of $P_n$ as nodes and choose weights $w_i$ so the rule is exact for all polynomials of degree $<n$. Any polynomial $p$ of degree $\le 2n-1$ can be written $p = qP_n + r$ with $\deg q,\deg r < n$. The $q P_n$ part integrates to zero by orthogonality and vanishes at the nodes, and the rule is exact on $r$ by construction, so the rule is exact on all of $p$. That is the degree of exactness $2n-1$: an $n$-point rule matches a $(2n)$-point Newton-Cotes rule in polynomial reach. The weights have the closed form

$$ w_i = \frac{2}{\left(1-x_i^2\right)\,\big[P_n'(x_i)\big]^2}. $$

In [ ]:
# ---------------------------------------------------------------------------
# Build Gauss-Legendre nodes and weights from the orthogonal polynomial
# ---------------------------------------------------------------------------
# We construct P_n symbolically with SymPy, take its roots as the nodes at
# high precision with mpmath, and apply the closed-form weight formula. This
# mirrors the theory directly rather than calling a black-box routine; the
# next cell checks the result against numpy's built-in for confidence.

def gauss_legendre(n):
    """Return (nodes, weights) for n-point Gauss-Legendre on [-1, 1].

    nodes are the roots of the Legendre polynomial P_n; weights use
    w_i = 2 / ((1 - x_i^2) P_n'(x_i)^2)."""
    x = sp.symbols("x")
    Pn = sp.legendre(n, x)                  # Legendre polynomial, exact
    Pn_prime = sp.diff(Pn, x)
    # high-precision roots of P_n
    roots = sp.nroots(Pn, n=mp.mp.dps)
    nodes, weights = [], []
    for r in roots:
        xi = mp.mpf(str(sp.re(r)))          # roots are real; drop tiny imag part
        deriv = mp.mpf(str(Pn_prime.subs(x, sp.nsimplify(r)).evalf(mp.mp.dps)))
        wi = 2 / ((1 - xi ** 2) * deriv ** 2)
        nodes.append(xi); weights.append(wi)
    order = np.argsort([float(v) for v in nodes])
    nodes   = [nodes[i]   for i in order]
    weights = [weights[i] for i in order]
    return nodes, weights

for n in (2, 3, 4):
    nd, wt = gauss_legendre(n)
    print(f"n = {n}")
    for xi, wi in zip(nd, wt):
        print(f"   node {float(xi): .6f}   weight {float(wi): .6f}")

In [ ]:
# ---------------------------------------------------------------------------
# Confirm the construction against numpy's built-in Gauss-Legendre
# ---------------------------------------------------------------------------
# numpy.polynomial.legendre.leggauss returns the same nodes and weights. Close
# agreement confirms our from-scratch construction is correct.
def check_against_numpy(n=5):
    nd, wt = gauss_legendre(n)
    nd_np, wt_np = np.polynomial.legendre.leggauss(n)
    dn = np.max(np.abs(np.array([float(v) for v in nd]) - nd_np))
    dw = np.max(np.abs(np.array([float(v) for v in wt]) - wt_np))
    print(f"n = {n}: max node difference {dn:.2e}, max weight difference {dw:.2e}")

check_against_numpy(5)

In [ ]:
# ---------------------------------------------------------------------------
# Verify the degree of exactness: an n-point rule is exact through 2n-1
# ---------------------------------------------------------------------------
# We integrate the monomials x^k on [-1, 1] and find the largest k for which
# the rule is exact. The exact integral is 0 for odd k and 2/(k+1) for even k.
def degree_of_exactness(n):
    nd, wt = gauss_legendre(n)
    nd = [mp.mpf(v) for v in nd]; wt = [mp.mpf(v) for v in wt]
    last_exact = -1
    for k in range(0, 2 * n + 2):
        approx = sum(w * x ** k for x, w in zip(nd, wt))
        exact  = mp.mpf(0) if k % 2 else mp.mpf(2) / (k + 1)
        if abs(approx - exact) < mp.mpf(10) ** (-25):
            last_exact = k
        else:
            break
    return last_exact

for n in (2, 3, 4, 5):
    print(f"n = {n}: exact through degree {degree_of_exactness(n)}  (theory {2*n - 1})")

In [ ]:
# ---------------------------------------------------------------------------
# Gauss vs composite Simpson at equal number of function evaluations
# ---------------------------------------------------------------------------
# On a smooth integrand Gauss quadrature converges far faster per evaluation.
# We integrate a smooth function on a general interval [a, b] by mapping the
# nodes from [-1, 1] with the affine change of variable x = (a+b)/2 + (b-a)/2 t.

def gauss_integrate(f, a, b, n):
    """n-point Gauss-Legendre estimate of the integral of f over [a, b]."""
    nd, wt = gauss_legendre(n)
    half = (b - a) / 2.0
    mid  = (a + b) / 2.0
    s = sum(float(w) * f(mid + half * float(t)) for t, w in zip(nd, wt))
    return half * s

def composite_simpson(f, a, b, n):
    """Composite Simpson on n panels (n even), used here for comparison."""
    if n % 2: n += 1
    x = np.linspace(a, b, n + 1); y = f(x); h = (b - a) / n
    return h / 3 * (y[0] + y[-1] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum())

def show_comparison(func="cos(3x)"):
    """Plot error vs number of function evaluations for Gauss and Simpson."""
    FUN = {
        "cos(3x)":   (lambda x: np.cos(3 * x), 0.0, 2.0),
        "exp(x)":    (np.exp,                  0.0, 2.0),
        "1/(1+25x^2)": (lambda x: 1 / (1 + 25 * x * x), -1.0, 1.0),
    }
    f, a, b = FUN[func]
    exact = float(mp.quad(lambda t: float(f(float(t))), [a, b]))
    evals = np.arange(2, 13)
    err_g = [abs(gauss_integrate(f, a, b, n) - exact) for n in evals]
    err_s = [abs(composite_simpson(f, a, b, n) - exact) for n in evals]
    plt.figure()
    plt.semilogy(evals, np.array(err_g) + 1e-18, "bo-", label="Gauss (n points)")
    plt.semilogy(evals, np.array(err_s) + 1e-18, "rs-", label="composite Simpson (n panels)")
    plt.xlabel("number of function evaluations"); plt.ylabel("absolute error")
    plt.title(f"Gauss vs Simpson on {func}")
    plt.legend(); plt.show()
    print(f"at ~12 evaluations: Gauss error {err_g[-1]:.2e}, Simpson error {err_s[-1]:.2e}")

show_comparison()

In [ ]:
# ---------------------------------------------------------------------------
# Interactive controls
# ---------------------------------------------------------------------------
interact(
    show_comparison,
    func=Dropdown(options=["cos(3x)", "exp(x)", "1/(1+25x^2)"], description="integrand"),
);

## Summary

- Gaussian quadrature chooses nodes and weights together, reaching degree of exactness $2n-1$ with only $n$ points.
- The optimal nodes are the roots of the Legendre polynomial $P_n$; the weights follow from a closed form and match numpy's built-in to machine precision.
- The $2n-1$ exactness is confirmed monomial by monomial.
- On smooth integrands Gauss quadrature reaches machine accuracy in a handful of evaluations, far ahead of composite Simpson at equal cost.